# Projet — Transfer Learning ResNet18 sur ESC-50 (sons d'animaux)

Ce notebook applique le **transfer learning** avec **ResNet18** pré-entraîné sur ImageNet pour classifier les mêmes 12 classes de sons d'animaux du dataset ESC-50.

On compare cette approche au CNN custom entraîné depuis zéro dans `entrainement_esc50.ipynb` (~68.8% de précision).

**Principe du transfer learning :**
- On part d'un ResNet18 qui a déjà appris à extraire des caractéristiques visuelles pertinentes sur 1.2 million d'images ImageNet.
- On gèle le backbone (toutes les couches convolutives) pour conserver ces connaissances.
- On remplace uniquement la dernière couche linéaire pour adapter le modèle à nos 12 classes.
- On entraîne seulement cette nouvelle tête — beaucoup moins de paramètres, convergence plus rapide.

## Imports

In [1]:
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import torchvision.models as models
import torchvision.transforms as T
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Chargement des métadonnées

Même dataset et même découpage que dans `entrainement_esc50.ipynb` :
- 12 classes animales, 480 échantillons au total
- Folds 1–4 en entraînement (384 échantillons), fold 5 en test (96 échantillons)

In [2]:
META_PATH = Path("meta/esc50.csv")
AUDIO_DIR = Path("audio")

ANIMAL_CLASSES = [
    "cat", "chirping_birds", "cow", "crickets", "crow",
    "dog", "frog", "hen", "insects", "pig", "rooster", "sheep"
]

df_full = pd.read_csv(META_PATH)
df = df_full[df_full["category"].isin(ANIMAL_CLASSES)].reset_index(drop=True)

print(f"Échantillons (total)   : {len(df_full)}")
print(f"Échantillons (animaux) : {len(df)}")
print(f"Nombre de classes      : {df['category'].nunique()}")
print(f"Classes                : {sorted(df['category'].unique())}")
df.head()

Échantillons (total)   : 2000
Échantillons (animaux) : 480
Nombre de classes      : 12
Classes                : ['cat', 'chirping_birds', 'cow', 'crickets', 'crow', 'dog', 'frog', 'hen', 'insects', 'pig', 'rooster', 'sheep']


,filename,fold,target,category,esc10,src_file,take
0,1-100032-A-0.wav,1,0,dog,True,100032,A
1,1-100038-A-14.wav,1,14,chirping_birds,False,100038,A
2,1-103298-A-9.wav,1,9,crow,False,103298,A
3,1-110389-A-0.wav,1,0,dog,True,110389,A
4,1-121951-A-8.wav,1,8,sheep,False,121951,A


## Extraction du spectrogramme Mel

Même pipeline que dans le notebook CNN — les spectrogrammes ont la forme `(128, 216)`.

Pour ResNet18, on devra ensuite :
1. **Répliquer le canal** : passer de `(1, 128, 216)` à `(3, 128, 216)` pour simuler une image RGB.
2. **Redimensionner** : passer à `(3, 224, 224)` — la taille attendue par ResNet18.
3. **Normaliser** avec les statistiques ImageNet.

In [3]:
def load_spectrogram(filename, sr=22050, n_mels=128, hop_length=512):
    """Charge un fichier WAV et retourne son spectrogramme Mel en dB."""
    path = AUDIO_DIR / filename
    signal, _ = librosa.load(path, sr=sr)
    mel = librosa.feature.melspectrogram(
        y=signal, sr=sr, n_mels=n_mels, hop_length=hop_length
    )
    mel_db = librosa.power_to_db(mel, ref=mel.max())
    return mel_db, sr, hop_length

sample = df.iloc[0]
spec, sr, hop = load_spectrogram(sample["filename"])
print(f"Forme du spectrogramme brut : {spec.shape}  (n_mels × trames temporelles)")
print(f"Classe                      : {sample['category']}")

d:\Projets\Code\IIM\cours_ia_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Forme du spectrogramme brut : (128, 216)  (n_mels × trames temporelles)
Classe                      : dog


## Dataset PyTorch — adaptation pour ResNet18

ResNet18 a été entraîné sur des images ImageNet de taille `(3, 224, 224)`. Nos spectrogrammes sont `(1, 128, 216)`. On adapte dans le `__getitem__` :

1. `unsqueeze(0)` → `(1, 128, 216)`
2. `.repeat(3, 1, 1)` → `(3, 128, 216)` : on copie le canal 3 fois pour simuler RGB
3. `T.Resize((224, 224))` → `(3, 224, 224)` : interpolation bilinéaire
4. `T.Normalize(ImageNet stats)` : normalisation avec la moyenne et l'écart-type d'ImageNet

> **Pourquoi les stats ImageNet ?** Le backbone ResNet18 a été entraîné avec ces valeurs. Utiliser la même normalisation assure que les activations de ses couches restent dans les plages attendues, même si l'entrée n'est pas une vraie photo.

In [4]:
resnet_transform = T.Compose([
    T.Resize((224, 224)),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class ESC50ResNetDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)
        categories = sorted(self.df["category"].unique())
        self.class_to_idx = {c: i for i, c in enumerate(categories)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spec, _, _ = load_spectrogram(row["filename"])              # (128, 216)
        x = torch.tensor(spec, dtype=torch.float32).unsqueeze(0)   # (1, 128, 216)
        x = x.repeat(3, 1, 1)                                       # (3, 128, 216)
        x = resnet_transform(x)                                      # (3, 224, 224)
        y = self.class_to_idx[row["category"]]
        return x, y


df_train = df[df["fold"] != 5]
df_test  = df[df["fold"] == 5]

train_dataset = ESC50ResNetDataset(df_train)
test_dataset  = ESC50ResNetDataset(df_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

x_batch, y_batch = next(iter(train_loader))
print(f"Train : {len(train_dataset)} échantillons  |  Test : {len(test_dataset)} échantillons")
print(f"Forme d'un batch : {x_batch.shape}  →  labels : {y_batch.shape}")

Train : 384 échantillons  |  Test : 96 échantillons
Forme d'un batch : torch.Size([32, 3, 224, 224])  →  labels : torch.Size([32])


## Construction du modèle — ResNet18 fine-tuné

**Stratégie :**
1. Charger ResNet18 avec les poids ImageNet pré-entraînés.
2. **Geler** tous les paramètres du backbone (`requires_grad = False`) — ils ne seront pas mis à jour.
3. **Remplacer** la couche finale `fc` (512 → 1000 classes ImageNet) par une nouvelle couche (512 → 12 classes animales).
4. Seule cette nouvelle tête est entraînable.

```
ResNet18 (backbone gelé)
  → conv1, layer1, layer2, layer3, layer4  [poids ImageNet fixés]
  → avgpool                                 [poids ImageNet fixés]
  → fc : Linear(512 → 12)                  [seule partie entraînable]
```

Avantage : au lieu d'entraîner **11.2M paramètres** depuis zéro, on n'entraîne que **6 156 paramètres** (512×12 + 12 biais).

In [5]:
NUM_CLASSES = 12

resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in resnet.parameters():
    param.requires_grad = False

resnet.fc = nn.Linear(512, NUM_CLASSES)

resnet = resnet.to(device)

n_trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
n_total     = sum(p.numel() for p in resnet.parameters())
print(f"Paramètres total      : {n_total:,}")
print(f"Paramètres entraînables : {n_trainable:,}  (tête uniquement)")

x_dummy = torch.randn(1, 3, 224, 224).to(device)
out = resnet(x_dummy)
print(f"Forward pass : entrée {tuple(x_dummy.shape)} → sortie {tuple(out.shape)}")

Paramètres total      : 11,182,668
Paramètres entraînables : 6,156  (tête uniquement)
Forward pass : entrée (1, 3, 224, 224) → sortie (1, 12)


## Entraînement

Même configuration que le CNN custom pour une comparaison équitable :
- **`CrossEntropyLoss`** : perte standard pour classification multi-classes
- **`AdamW`** (lr=1e-3) : on optimise uniquement `resnet.fc.parameters()`
- **`ReduceLROnPlateau`** : réduit le LR si la validation stagne (patience=3, factor=0.5)
- **Early stopping** (patience=5) : arrête si pas d'amélioration, restaure les meilleurs poids
- **TensorBoard** : logs dans `runs/resnet18_esc50/`

In [6]:
def train_loop(dataloader, model, loss_fn, optimizer, writer, epoch):
    model.train()
    pbar = tqdm(dataloader, desc="Train")
    for batch, (x, y_true) in enumerate(pbar):
        x, y_true = x.to(device), y_true.to(device)
        y_pred = model(x)
        loss = loss_fn(y_pred, y_true)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        pbar.set_postfix(loss=f"{loss.item():.4f}")
        writer.add_scalar("Loss/train", loss.item(), epoch * len(dataloader) + batch)


def test_loop(dataloader, model, loss_fn, optimizer, writer, epoch):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for x, y_true in tqdm(dataloader, desc="Test "):
            x, y_true = x.to(device), y_true.to(device)
            y_pred = model(x)
            test_loss += loss_fn(y_pred, y_true).item()
            correct += (y_pred.argmax(1) == y_true).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Accuracy : {100 * correct:.1f}%  |  Avg loss : {test_loss:.6f}")

    writer.add_scalar("Loss/test", test_loss, epoch)
    writer.add_scalar("Accuracy/test", 100 * correct, epoch)
    writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch)

    return test_loss, correct

In [7]:
criterion  = nn.CrossEntropyLoss()
optimizer  = torch.optim.AdamW(resnet.fc.parameters(), lr=1e-3)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
writer     = SummaryWriter("runs/resnet18_esc50")

In [8]:
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns

CLASS_NAMES = sorted(df["category"].unique())

epochs      = 30
es_patience = 5

best_loss    = float("inf")
best_acc     = 0.0
best_weights = None
epochs_without_improvement = 0

for t in range(epochs):
    print(f"Epoch : {t+1}  |  lr = {optimizer.param_groups[0]['lr']:.2e}")
    train_loop(train_loader, resnet, criterion, optimizer, writer, t)
    val_loss, val_acc = test_loop(test_loader, resnet, criterion, optimizer, writer, t)

    scheduler.step(val_loss)

    if val_loss < best_loss:
        best_loss    = val_loss
        best_acc     = val_acc
        best_weights = {k: v.cpu().clone() for k, v in resnet.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        print(f"  (pas d'amélioration depuis {epochs_without_improvement} époque(s))")
        if epochs_without_improvement >= es_patience:
            print(f"Early stopping déclenché à l'époque {t+1}.")
            break

resnet.load_state_dict(best_weights)

# Matrice de confusion finale dans TensorBoard
resnet.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x, y_true in test_loader:
        x = x.to(device)
        all_preds.extend(resnet(x).argmax(1).cpu().tolist())
        all_labels.extend(y_true.tolist())

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Prédit")
ax.set_ylabel("Réel")
ax.set_title("Matrice de confusion — ResNet18 ESC-50 (fold 5)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
writer.add_figure("Confusion Matrix", fig, global_step=t)
plt.show()

writer.close()
print(f"\nEntraînement terminé. Meilleure précision : {100 * best_acc:.1f}%")

Epoch : 1  |  lr = 1.00e-03


Test : 100%|██████████| 3/3 [00:03<00:00,  1.15s/it]


Accuracy : 10.4%  |  Avg loss : 2.468708
Epoch : 2  |  lr = 1.00e-03


Test : 100%|██████████| 3/3 [00:02<00:00,  1.16it/s]


Accuracy : 29.2%  |  Avg loss : 2.028068
Epoch : 3  |  lr = 1.00e-03


...

Accuracy : 78.1%  |  Avg loss : 0.862441
Epoch : 29  |  lr = 1.00e-03


Test : 100%|██████████| 3/3 [00:03<00:00,  1.09s/it]


Accuracy : 80.2%  |  Avg loss : 0.839197
Epoch : 30  |  lr = 1.00e-03


Test : 100%|██████████| 3/3 [00:03<00:00,  1.16s/it]


Accuracy : 76.0%  |  Avg loss : 0.846709
  (pas d'amélioration depuis 1 époque(s))

Entraînement terminé. Meilleure précision : 80.2%


## Export ONNX

**ONNX** (Open Neural Network Exchange) est un format ouvert qui permet de déployer un modèle PyTorch dans n'importe quel environnement compatible : moteurs d'inférence haute performance (ONNX Runtime, TensorRT), applications mobiles, navigateurs (via ONNX.js), etc.

L'export fonctionne par **traçage** : PyTorch effectue un forward pass avec un tenseur factice (*dummy input*) et enregistre toutes les opérations dans un graphe statique.

Points clés :
- Le modèle doit être en mode **`eval()`** avant l'export pour désactiver Dropout et BatchNorm en mode entraînement.
- `input_names` et `output_names` donnent des noms lisibles aux entrées/sorties du graphe.
- `dynamic_axes` permet d'accepter des batches de taille variable à l'inférence (`batch_size` n'est pas fixé dans le graphe).
- `opset_version=17` cible une version ONNX récente et largement supportée.

In [9]:
import torch
from pathlib import Path

ONNX_DIR  = Path("onnx")
ONNX_DIR.mkdir(exist_ok=True)
ONNX_PATH = ONNX_DIR / "resnet18_esc50.onnx"

resnet.eval()

dummy_input = torch.randn(1, 3, 224, 224).to(device)

torch.onnx.export(
    resnet,
    dummy_input,
    str(ONNX_PATH),
    opset_version=17,
    input_names=["spectrogram"],
    output_names=["logits"],
    dynamic_axes={
        "spectrogram": {0: "batch_size"},
        "logits":      {0: "batch_size"},
    },
)

print(f"Modèle exporté : {ONNX_PATH}")

C:\Users\arnau\AppData\Local\Temp\ipykernel_29080\4205871254.py:12: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0611 12:05:52.655000 29080 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0611 12:05:53.302000 29080 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Users\arnau\AppData\Local\Python\pythoncore-3.14-64\Lib\copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).
Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "d:\Projets\Code\IIM\cours_ia_2\.venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "d:\Projets\Code\IIM\cours_ia_2\.venv\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "d:\Projets\Code\IIM\cours_ia_2\.venv\Lib\site

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Modèle exporté : onnx\resnet18_esc50.onnx


## Comparaison des résultats

| Modèle | Paramètres entraînables | Précision test (fold 5) |
|--------|------------------------|-------------------------|
| CNN custom (depuis zéro) | 14 252 236 | ~68.8% |
| ResNet18 (tête seule) | 6 156 | à compléter |

**Observations attendues :**
- Le ResNet18 fine-tuné converge beaucoup plus vite (moins d'époques nécessaires).
- Avec seulement 6 156 paramètres à entraîner, le risque de sur-apprentissage est très réduit.
- La précision devrait être supérieure (~75–85%) grâce aux représentations apprises sur ImageNet, qui captent des structures locales réutilisables même pour des spectrogrammes.